# CSPF — stage H: controlled head-to-head with existing perturbation generators (+ sheets for a blind human check and a second audit round)
Reviewer point: *the comparison with CheckList / TextFlint / NL-Augmenter is only structural.* This stage runs **CSPF and six widely used generators on the same source tweets, with the same classifiers**, and compares them on what matters for the paper's claims:
- **H1** how many pairs each generator produces, how often NLI judges them equivalent (nli_bi ≥ 0.9), and the flip rate overall / on NLI-equivalent pairs / on NLI-non-equivalent pairs (review-cluster bootstrap CIs).
- **H2** do the generators rank the classifiers the same way (Spearman between per-model flip rates)?
- **H3** *budget-matched coverage*: for each review, take the NLI-equivalent pairs of the baselines and the **same number** of NLI-equivalent CSPF pairs; how often does CSPF expose a model flip that **none** of the baselines exposes (and vice versa)?
- **H4** a shuffled, blind sheet (baseline pairs + CSPF pairs, generator hidden) for a human meaning check, and **H5** a second-round audit sheet (family × severity stratified, with the clarified guidelines).
Generators (all run here): keyboard-typo noise, WordNet synonym substitution (nlpaug), BERT contextual substitution (nlpaug), random swap/delete (EDA-style, nlpaug), T5 paraphrase, back-translation (Marian en→de→en).
Needs a GPU (T4 is enough, ≈ 40–60 min) and `ALL_MODELS_combined_FIXED.csv`. Output: `CSPF_stageH_bundle.zip` (send it to me) and the two sheets.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/postdoc-papers/paper2"
FIXED_CSV = f"{BASE}/ALL_MODELS_combined_FIXED.csv"; A1_AUDIT = f"{BASE}/annotated_output.csv"
OUT = f"{BASE}/results_FIXED/reviewer_H"; N_REVIEWS = 2000; SEED = 0; B_REP = 500; MIN_AGREE = 0.97; N_HUMAN_PER_GEN = 20; N_HUMAN_CSPF = 40; N_ROUND2_PER_FAMILY = 24
CLEAN_EXCLUDE = {"semeval13", "semeval16", "semeval17", "debate08", "hcr"}          # exposed datasets (Section 4.3) are not used here
import os, subprocess, sys, warnings, zipfile, glob, random, re, hashlib; warnings.filterwarnings("ignore"); os.makedirs(OUT, exist_ok=True)
subprocess.run([sys.executable,"-m","pip","-q","install","nlpaug","sacremoses","sentencepiece"],check=False)
import numpy as np, pandas as pd
from scipy.stats import spearmanr
rng = np.random.default_rng(SEED); random.seed(SEED)
GEN_ORDER = ["typo_keyboard", "wordnet_synonym", "bert_contextual", "eda_swap_delete", "t5_paraphrase", "back_translation"]
FAMS_PROP = ["P02_discourse", "P08_synonym", "P10_syntax", "P11_entity", "P12_register"]; FAMS_MOD = ["P03_negation", "P04_sarcasm", "P05_dilution", "P07_contrastive", "P09_intensity"]
GOLD = [  # invented rows with unambiguous answers (meaning, sentiment, natural); None = not checked. Mixed into both sheets to check that annotators read the rows.
 ("The movie was great.", "That movie was great.", "yes", "yes", "yes"), ("The service was terrible.", "The service was awful.", "yes", "yes", "yes"),
 ("The movie was great.", "The movie was terrible.", "no", "no", "yes"), ("I loved the concert last night.", "I loved the football game last night.", "no", "yes", "yes"),
 ("The food was awful.", "The food was kind of awful, I guess.", "partial", "yes", "yes"), ("The phone is amazing.", "The phone is the amazing and of the the.", None, "yes", "no")]
REGISTRY = {"RoBERTa":"Etelis/Sentiment140_roBERTa_5E","XLNet":"Etelis/Sentiment140_XLNET_5E","ELECTRA":"Etelis/Sentiment140_ELECTRA_5E","DistilBERT":"Etelis/Sentiment140_DistilBERT_5E","ALBERT":"Etelis/Sentiment140_ALBERT_5E","BERT":"Etelis/Sentiment140_BERT_5E",
  "SiEBERT-large":"siebert/sentiment-roberta-large-english","TwRoBERTa-latest":"cardiffnlp/twitter-roberta-base-sentiment-latest","BERTweet":"finiteautomata/bertweet-base-sentiment-analysis","XLM-R-Twitter":"cardiffnlp/twitter-xlm-roberta-base-sentiment",
  "DistilBERT-SST2":"distilbert-base-uncased-finetuned-sst-2-english","RoBERTa-SST2":"textattack/roberta-base-SST-2","DistilBERT-IMDb":"lvwerra/distilbert-imdb","ModernBERT-sent":"clapAI/modernBERT-base-multilingual-sentiment"}   # GPT2 and T5 need special decoding and are left out

In [ ]:
# ── H0: shared helpers (corpus, NLI, classifier predictions) ──
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
dev = "cuda" if torch.cuda.is_available() else "cpu"; print("device:", dev)
CORPUS_DIR = "/content/CSPF"
def load_corpus(cols):
    if not os.path.exists(CORPUS_DIR): subprocess.run(["git","clone","--depth","1","https://github.com/hadis-1/CSPF",CORPUS_DIR], check=True)
    fs = sorted(glob.glob(f"{CORPUS_DIR}/datasets/*_perturbed.csv")); assert fs, "corpus files not found"
    d = pd.concat([pd.read_csv(f, usecols=cols, low_memory=False) for f in fs], ignore_index=True); d["key"] = d.source_dataset.astype(str)+"::"+d.review_idx.astype(str); return d
_NLI = {}
@torch.no_grad()
def nli_bi(orig, pert, bs=64, name="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"):
    if "m" not in _NLI:
        _NLI["t"] = AutoTokenizer.from_pretrained(name); m = AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval(); _NLI["m"] = m.half() if dev == "cuda" else m
        _NLI["e"] = {v.lower(): k for k, v in _NLI["m"].config.id2label.items()}["entailment"]
    t, m, e = _NLI["t"], _NLI["m"], _NLI["e"]
    def run(P, H):
        out = np.zeros(len(P), dtype=np.float32); order = np.argsort([len(a)+len(b) for a, b in zip(P, H)])
        for i in range(0, len(order), bs):
            ix = order[i:i+bs]; enc = t([P[j] for j in ix], [H[j] for j in ix], padding=True, truncation=True, max_length=160, return_tensors="pt").to(dev)
            out[ix] = torch.softmax(m(**enc).logits.float(), -1)[:, e].cpu().numpy()
        return out
    return np.minimum(run(list(orig), list(pert)), run(list(pert), list(orig)))
def _neg_pos_idx(id2label):
    n = len(id2label); names = {int(k): str(v).lower() for k, v in id2label.items()}
    neg = [k for k, v in names.items() if v.startswith("neg") or v in ("label_0", "0")]; pos = [k for k, v in names.items() if v.startswith("pos") or v in ("1",) or (v == "label_1" and n == 2) or (v == "label_2" and n == 3)]
    if not neg or not pos: neg, pos = [0], [n-1]
    return neg[0], pos[0]
@torch.no_grad()
def predict(model_id, texts, bs=64):
    """label per text, 1 = positive; argmax restricted to the negative and positive logits (as in the paper)"""
    t = AutoTokenizer.from_pretrained(model_id); m = AutoModelForSequenceClassification.from_pretrained(model_id).to(dev).eval()
    if dev == "cuda": m = m.half()
    ni, pi = _neg_pos_idx(m.config.id2label); out = np.zeros(len(texts), dtype=np.int8); order = np.argsort([len(x) for x in texts])
    for i in range(0, len(order), bs):
        ix = order[i:i+bs]; enc = t([texts[j] for j in ix], padding=True, truncation=True, max_length=128, return_tensors="pt").to(dev)
        lg = m(**enc).logits.float(); out[ix] = (lg[:, pi] > lg[:, ni]).long().cpu().numpy()
    del m; torch.cuda.empty_cache(); return out
def norm_label(x):
    s = str(x).strip().lower(); return 1 if (s.startswith("pos") or s in ("1", "label_1", "label_2", "4")) else 0

In [ ]:
# ── H1a: sample reviews from the unexposed datasets; load CSPF pairs and the stored predictions for them ──
cols = ["review_idx","source_dataset","original_review","perturbed_review","perturbation_family","severity"]
corp = load_corpus(cols); corp = corp[~corp.source_dataset.astype(str).str.lower().isin(CLEAN_EXCLUDE)]
U = corp.drop_duplicates("key")[["key","source_dataset","review_idx","original_review"]]; U = U.sample(min(N_REVIEWS, len(U)), random_state=SEED).reset_index(drop=True)
corp = corp[corp.key.isin(set(U.key))].reset_index(drop=True); print(len(U), "reviews;", len(corp), "CSPF pairs on them; datasets:", U.source_dataset.nunique())
hdr = pd.read_csv(FIXED_CSV, nrows=0).columns.tolist(); AVAIL = sorted({c[:-len("_pert_label")] for c in hdr if c.endswith("_pert_label")})
MODELS0 = [m for m in AVAIL if m in REGISTRY]; print("models in the prediction file that this stage can re-run:", MODELS0, "| skipped:", sorted(set(AVAIL)-set(MODELS0)))
meta = ["source_dataset","review_idx","perturbed_review","perturbation_family"]; use = meta + [f"{m}_{s}_label" for m in MODELS0 for s in ("orig","pert")]; keys = set(U.key); parts = []
for ch in pd.read_csv(FIXED_CSV, usecols=use, chunksize=100000, low_memory=False):
    ch["key"] = ch.source_dataset.astype(str)+"::"+ch.review_idx.astype(str); ch = ch[ch.key.isin(keys)]
    if len(ch): parts.append(ch)
FX = pd.concat(parts, ignore_index=True); FX["source_dataset"] = FX.source_dataset.astype(str)
corp["source_dataset"] = corp.source_dataset.astype(str); J = corp.merge(FX.drop(columns=["review_idx","key"]), on=["source_dataset","perturbed_review","perturbation_family"], how="inner").drop_duplicates(["key","perturbed_review","perturbation_family","severity"]); print(len(J), "CSPF pairs matched to stored predictions")
ORIG = FX.drop_duplicates("key").set_index("key")

In [ ]:
# ── H1b: re-run the classifiers on the original texts and keep only models that reproduce the stored labels (>= MIN_AGREE) ──
MODELS, agree, P0 = [], {}, {}
for m in MODELS0:
    try:
        p = predict(REGISTRY[m], U.original_review.tolist()); P0[m] = p; st = ORIG.loc[U.key, f"{m}_orig_label"].map(norm_label).values; agree[m] = float((p == st).mean())
        print(f"{m:18s} agreement with stored labels on original texts: {agree[m]:.3f}", "" if agree[m] >= MIN_AGREE else " -> dropped")
        if agree[m] >= MIN_AGREE: MODELS.append(m)
    except Exception as e: print(m, "failed:", str(e)[:100], "-> dropped")
assert len(MODELS) >= 5, "fewer than 5 models reproduce the stored labels"; pd.Series(agree).to_csv(f"{OUT}/H1_model_agreement_check.csv"); print("models used:", MODELS)
for m in MODELS: J[f"flip_{m}"] = (J[f"{m}_orig_label"].map(norm_label).values != J[f"{m}_pert_label"].map(norm_label).values).astype(np.int8)

In [ ]:
# ── H1c: baseline generators on the same source tweets ──
def gen_typo(texts):
    import nlpaug.augmenter.char as nac; a = nac.KeyboardAug(aug_char_p=0.1, aug_word_p=0.15, aug_word_min=1, include_special_char=False, include_numeric=False); return _aug(a, texts)
def gen_wordnet(texts):
    import nlpaug.augmenter.word as naw; a = naw.SynonymAug(aug_src="wordnet", aug_p=0.15); return _aug(a, texts)
def gen_bert(texts):
    import nlpaug.augmenter.word as naw; a = naw.ContextualWordEmbsAug(model_path="distilbert-base-uncased", action="substitute", aug_p=0.15, device=dev); return _aug(a, texts)
def gen_eda(texts):
    import nlpaug.augmenter.word as naw; sw = naw.RandomWordAug(action="swap", aug_p=0.15); dl = naw.RandomWordAug(action="delete", aug_p=0.15)
    return [ (_aug(sw, [t])[0] if i % 2 == 0 else _aug(dl, [t])[0]) for i, t in enumerate(texts)]
def _aug(a, texts):
    out = []
    for t in texts:
        try: r = a.augment(t); r = r[0] if isinstance(r, list) else r
        except Exception: r = t
        out.append(r if isinstance(r, str) and r.strip() else t)
    return out
def gen_t5(texts):
    from transformers import AutoModelForSeq2SeqLM
    nm = "humarin/chatgpt_paraphraser_on_T5_base"; t = AutoTokenizer.from_pretrained(nm); m = AutoModelForSeq2SeqLM.from_pretrained(nm).to(dev).eval(); outs = []
    for i in range(0, len(texts), 32):
        enc = t([f"paraphrase: {x}" for x in texts[i:i+32]], return_tensors="pt", padding=True, truncation=True, max_length=128).to(dev)
        with torch.no_grad(): g = m.generate(**enc, num_beams=4, max_length=128, repetition_penalty=1.2, no_repeat_ngram_size=3)
        outs += t.batch_decode(g, skip_special_tokens=True)
    del m; torch.cuda.empty_cache(); return outs
def gen_bt(texts):
    from transformers import MarianMTModel, MarianTokenizer
    def tr(name, xs):
        tk = MarianTokenizer.from_pretrained(name); mm = MarianMTModel.from_pretrained(name).to(dev).eval(); o = []
        for i in range(0, len(xs), 32):
            enc = tk(xs[i:i+32], return_tensors="pt", padding=True, truncation=True, max_length=128).to(dev)
            with torch.no_grad(): g = mm.generate(**enc, num_beams=4, max_length=128)
            o += tk.batch_decode(g, skip_special_tokens=True)
        del mm; torch.cuda.empty_cache(); return o
    return tr("Helsinki-NLP/opus-mt-de-en", tr("Helsinki-NLP/opus-mt-en-de", texts))
GENS = {"typo_keyboard": gen_typo, "wordnet_synonym": gen_wordnet, "bert_contextual": gen_bert, "eda_swap_delete": gen_eda, "t5_paraphrase": gen_t5, "back_translation": gen_bt}

In [ ]:
rows = []; texts = U.original_review.tolist()
for g in GEN_ORDER:
    f = f"{OUT}/H_gen_{g}.csv"
    if os.path.exists(f): R = pd.read_csv(f)
    else:
        random.seed(SEED); np.random.seed(SEED); R = pd.DataFrame({"key": U.key, "orig": texts, "pert": GENS[g](texts)}); R.to_csv(f, index=False)
    R["gen"] = g; R["changed"] = (R.orig.str.strip() != R.pert.astype(str).str.strip()); print(f"{g:18s} changed {R.changed.mean():.1%} of the tweets"); rows.append(R)
BG = pd.concat(rows, ignore_index=True); BG["pert"] = BG.pert.astype(str); BGc = BG[BG.changed].reset_index(drop=True)

In [ ]:
# ── H1d: NLI for baseline pairs and for the CSPF pairs on the same reviews; classifier predictions on the baseline pairs ──
oi = U.set_index("key").original_review
BGc["nli"] = nli_bi(BGc.orig, BGc.pert); J["orig"] = J.key.map(oi).values; J["nli"] = nli_bi(J.orig, J.perturbed_review)
P0 = pd.DataFrame({m: P0[m] for m in MODELS}, index=U.key)
for m in MODELS:
    p = predict(REGISTRY[m], BGc.pert.tolist()); BGc[f"flip_{m}"] = (p != P0.loc[BGc.key, m].values).astype(np.int8)
BGc.to_csv(f"{OUT}/H_baseline_pairs_scored.csv", index=False); J[["key","perturbation_family","severity","nli"]+[f"flip_{m}" for m in MODELS]].to_csv(f"{OUT}/H_cspf_pairs_scored.csv", index=False)

In [ ]:
# ── H1/H2: per-generator summary and model rankings ──
FL = [f"flip_{m}" for m in MODELS]
def add_group(name, df, rows):
    df = df.copy(); df["y"] = df[FL].mean(1); rows.append((name, df))
G = []
for g in GEN_ORDER: add_group(g, BGc[BGc.gen == g], G)
add_group("CSPF all families", J, G); add_group("CSPF propositional families", J[J.perturbation_family.isin(FAMS_PROP)], G); add_group("CSPF meaning-modulating families", J[J.perturbation_family.isin(FAMS_MOD)], G); add_group("CSPF hedging + pragmatic", J[J.perturbation_family.isin(["P01_hedging","P06_pragmatic"])], G)
def cboot(df, fn, B=B_REP):
    uk, ki = np.unique(df.key.values, return_inverse=True); nc = len(uk); v = []
    for _ in range(B):
        c = np.bincount(rng.integers(0, nc, nc), minlength=nc)[ki].astype(float)
        if c.sum() > 0: v.append(fn(df, c))
    return np.percentile(v, 2.5), np.percentile(v, 97.5)
wmean = lambda col, mask=None: (lambda df, c: (c*df[col].values*(1 if mask is None else mask(df))).sum()/max((c*(1 if mask is None else mask(df))).sum(), 1e-9))
rows = []
for name, df in G:
    eq = lambda d: (d.nli.values >= 0.9).astype(float); ne = lambda d: (d.nli.values < 0.5).astype(float)
    r = dict(source=name, pairs=len(df), reviews=df.key.nunique(), nli_mean=round(df.nli.mean(), 3), share_nli_ge_0_9=round((df.nli >= 0.9).mean(), 3), flip_all=round(df.y.mean(), 4))
    r["flip_all_lo"], r["flip_all_hi"] = [round(x, 4) for x in cboot(df, wmean("y"))]
    for lab, mk in [("equiv", eq), ("nonequiv", ne)]:
        s = mk(df) > 0
        if s.sum() >= 30:
            sub = df[s]; r[f"flip_{lab}"] = round(sub.y.mean(), 4); r[f"flip_{lab}_n"] = int(s.sum()); lo, hi = cboot(sub, wmean("y")); r[f"flip_{lab}_lo"], r[f"flip_{lab}_hi"] = round(lo, 4), round(hi, 4)
    rows.append(r)
H1 = pd.DataFrame(rows); print(H1.to_string(index=False)); H1.to_csv(f"{OUT}/H1_generator_summary.csv", index=False)
pm = {name: df[df.nli >= 0.9][FL].mean().values for name, df in G if (df.nli >= 0.9).sum() >= 30}; names = list(pm); rho = pd.DataFrame([[spearmanr(pm[a], pm[b])[0] for b in names] for a in names], index=names, columns=names).round(2)
print("\nSpearman between per-model flip rates on NLI-equivalent pairs:"); print(rho.to_string()); rho.to_csv(f"{OUT}/H2_model_rank_correlation.csv"); pd.DataFrame({n: pm[n] for n in names}, index=MODELS).round(4).to_csv(f"{OUT}/H2_per_model_flip_equiv.csv")

In [ ]:
# ── H3: budget-matched coverage — flips that CSPF exposes and no baseline exposes (and vice versa), on NLI-equivalent pairs only ──
EQ = 0.9; R_DRAW = 20; be = BGc[BGc.nli >= EQ]; ce = J[J.nli >= EQ].reset_index(drop=True); cases = []
bk = {k: v for k, v in be.groupby("key").indices.items()}; ck = {k: v for k, v in ce.groupby("key").indices.items()}; BF = be[FL].values; CF = ce[FL].values
for k in bk:
    if k not in ck: continue
    nb = len(bk[k]); 
    if len(ck[k]) < nb: continue
    B = BF[bk[k]].max(0); S = np.mean([CF[rng.choice(ck[k], nb, replace=False)].max(0) for _ in range(R_DRAW)], 0)
    cases.append((k, nb, B, S))
print(len(cases), "reviews with >= 1 NLI-equivalent baseline pair and at least as many NLI-equivalent CSPF pairs")
Bm = np.array([c[2] for c in cases]); Sm = np.array([c[3] for c in cases]); ks = np.array([c[0] for c in cases])
def stats(ix):
    b, s = Bm[ix], Sm[ix]; return dict(baseline_exposes=b.mean(), cspf_exposes=s.mean(), cspf_only=(s*(1-b)).mean(), baseline_only=((1-s)*b).mean(), both=(s*b).mean(), cspf_only_share_of_cspf=(s*(1-b)).sum()/max(s.sum(), 1e-9))
est = stats(np.arange(len(cases))); bsd = []
for _ in range(B_REP): bsd.append(stats(rng.integers(0, len(cases), len(cases))))
bsd = pd.DataFrame(bsd); H3 = pd.DataFrame([dict(quantity=q, estimate=round(v, 4), ci_lo=round(bsd[q].quantile(.025), 4), ci_hi=round(bsd[q].quantile(.975), 4)) for q, v in est.items()]); print(H3.to_string(index=False)); H3.to_csv(f"{OUT}/H3_budget_matched_coverage.csv", index=False)
pm_rows = []
for i, m in enumerate(MODELS): pm_rows.append(dict(model=m, baseline_exposes=round(Bm[:, i].mean(), 4), cspf_exposes=round(Sm[:, i].mean(), 4), cspf_only=round((Sm[:, i]*(1-Bm[:, i])).mean(), 4)))
pd.DataFrame(pm_rows).to_csv(f"{OUT}/H3_per_model.csv", index=False); print(pd.DataFrame(pm_rows).to_string(index=False))

In [ ]:
# ── H4: blind human-check sheet (baseline pairs + CSPF pairs, generator hidden) ──
def draw(df, n): return df.sample(min(n, len(df)), random_state=SEED)
sh = []
for g in GEN_ORDER: x = draw(BGc[BGc.gen == g], N_HUMAN_PER_GEN); sh.append(pd.DataFrame({"source": g, "family": "", "original_review": x.orig.values, "perturbed_review": x.pert.values, "key": x.key.values}))
Jc = J.sample(frac=1, random_state=SEED); x = Jc[Jc.groupby("perturbation_family").cumcount() < int(np.ceil(N_HUMAN_CSPF/12))].head(N_HUMAN_CSPF)
sh.append(pd.DataFrame({"source": "CSPF", "family": x.perturbation_family.values, "original_review": x.orig.values, "perturbed_review": x.perturbed_review.values, "key": x.key.values}))
gold = pd.DataFrame({"source": "GOLD", "family": "", "original_review": [g[0] for g in GOLD], "perturbed_review": [g[1] for g in GOLD], "key": "", "exp_meaning": [g[2] for g in GOLD], "exp_sentiment": [g[3] for g in GOLD], "exp_natural": [g[4] for g in GOLD]}); sh.append(gold)
S4 = pd.concat(sh, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True); S4["sample_id"] = [f"HB_{i+1:03d}" for i in range(len(S4))]
S4[["sample_id","source","family","key","exp_meaning","exp_sentiment","exp_natural"]].to_csv(f"{OUT}/H4_blind_sheet_KEY_do_not_share.csv", index=False)
S4s = S4[["sample_id","original_review","perturbed_review"]].copy()
for c in ["meaning_a","sentiment_a","natural_a","comment_a"]: S4s[c] = ""
S4s.to_csv(f"{OUT}/H4_blind_sheet_for_annotator.csv", index=False, encoding="utf-8-sig"); print(len(S4s), "pairs in the blind sheet")

In [ ]:
# ── H5: second-round audit sheet (family x severity stratified; excludes reviews audited in round 1) ──
full = load_corpus(["review_idx","source_dataset","original_review","perturbed_review","perturbation_family","severity","position_strategy","variant_label"])
if os.path.exists(A1_AUDIT):
    a1 = pd.read_csv(A1_AUDIT, encoding="utf-8-sig"); done = set(a1.source_dataset.astype(str)+"::"+a1.review_idx.astype(str)); full = full[~full.key.isin(done)]; print("excluded round-1 reviews:", len(done))
full["sev"] = full.severity.astype(str); picks = []
for f, g in full.groupby("perturbation_family"):
    per = int(np.ceil(N_ROUND2_PER_FAMILY/3)); x = g.groupby("sev", group_keys=False).apply(lambda d: d.sample(min(per, len(d)), random_state=SEED)); picks.append(x.sample(min(N_ROUND2_PER_FAMILY, len(x)), random_state=SEED))
R2 = pd.concat(picks); R2["source"] = "CSPF"
gold2 = pd.DataFrame({"source": "GOLD", "original_review": [g[0] for g in GOLD], "perturbed_review": [g[1] for g in GOLD], "exp_meaning": [g[2] for g in GOLD], "exp_sentiment": [g[3] for g in GOLD], "exp_natural": [g[4] for g in GOLD]})
R2 = pd.concat([R2, gold2], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True); R2["sample_id"] = [f"R2_{i+1:03d}" for i in range(len(R2))]
R2[["sample_id","source","source_dataset","review_idx","perturbation_family","severity","position_strategy","variant_label","exp_meaning","exp_sentiment","exp_natural"]].to_csv(f"{OUT}/H5_round2_KEY_do_not_share.csv", index=False)
R2s = R2[["sample_id","original_review","perturbed_review"]].copy()
for c in ["meaning_a","sentiment_a","natural_a","comment_a"]: R2s[c] = ""
R2s.to_csv(f"{OUT}/H5_round2_sheet_for_annotators.csv", index=False, encoding="utf-8-sig"); print(len(R2s), "pairs in the round-2 sheet")
with zipfile.ZipFile(f"{OUT}/CSPF_stageH_bundle.zip","w",zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/*.csv"):
        if "KEY_do_not_share" in f or "sheet_for" in f or "H_gen_" in f: continue
        z.write(f, os.path.basename(f))
print("send me:", f"{OUT}/CSPF_stageH_bundle.zip", "\nsheets for annotators:", f"{OUT}/H4_blind_sheet_for_annotator.csv", f"{OUT}/H5_round2_sheet_for_annotators.csv")
try:
    from google.colab import files; files.download(f"{OUT}/CSPF_stageH_bundle.zip")
except Exception as e: print("download manually from Drive:", e)